In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ----------------------------------
# 날짜 입력받기
#   -> 입력하지 않으면 오늘날짜 반환
#   -> 잘못된 형식이면 None 반환
# ----------------------------------
def date_input():

    # 날짜 입력받기
    date_str = input("날짜:(형식:20200105)")
    
    # 날짜를 입력하지 않으면 오늘 날짜 가져오기
    if date_str=="":
        date_str = datetime.now().strftime('%Y%m%d')
    
    # 날짜 형식 변환 (YYYYmmdd --> YYYY-mm-dd)
    try:
        date_str = datetime.strptime(date_str, '%Y%m%d').strftime('%Y-%m-%d')
    except:
        print("날짜 형식이 올바르지 않습니다.")
        date_str = None
    
    return date_str

date = date_input()

print(f'입력한 날짜 : {date}')


입력한 날짜 : 2025-11-03


In [59]:
page = 1   
data_list = []

while True:
    # ----------------------------------
    # 웹페이지 요청하여 응답객체 받기
    # ----------------------------------
    url = f'https://finance.naver.com/news/mainnews.naver?date={date}&page={page}'
    response = requests.get(url)
    html = response.text

    # ----------------------------------
    # 응답받은 웹페이지 파싱하여 BeautifulSoup 객체 생성
    # ----------------------------------
    soup = BeautifulSoup(html, 'html.parser')

    # ----------------------------------
    # article 리스트 추출
    # ----------------------------------   
    articles = soup.select('.block1')
    articles

    # ----------------------------------
    # article 리스트에서 요소의 텍스트, 속성 추출
    # ----------------------------------   
    for article in articles: 
        subject = article.select_one('.articleSubject > a').text
        summary = article.select_one('.articleSummary').text.split(' ')[:-3]
        summary = ' '.join(summary).strip()
        
        press = article.select_one('.press').text
        wdate = article.select_one('.wdate').text
        link = article.select_one('.articleSubject > a').attrs['href']
        data_list.append({'subject':subject, 
                        'summary':summary, 
                        'press': press, 
                        'wdate': wdate, 
                        'link': link})

    # ----------------------------------
    # 페이지 이동
    # ----------------------------------   
    if soup.select_one('.pgRR'): #pgRR이 있다면 실행 / 마지막 장에는 pgRR이 없음
        print(page)
        page+=1
    else:
        break

    time.sleep(2) # 봇으로 인식되면 차단당할 수 있고, 어느정도 딜레이를 줘야 페이지가 모두 로드되어 크롤링 할 수 있음



1
2
3
4


In [55]:
# ---------------------------
# 데이터프레임 생성
# ---------------------------
df = pd.DataFrame(data_list)

In [58]:
# ------------
# csv 파일로 저장
# ------------
df.to_csv(f'data/네이버증권뉴스({date}).csv')

In [57]:
# ------------
# excel 파일로 저장
# ------------
df.to_excel(f'data/네이버증권뉴스({date}).xlsx')